In [0]:
trade = spark.table("bronze_trade_flows")


In [0]:
display(trade)

In [0]:
de_2605 = trade.filter("(reporterCode = 276 OR partnerCode = 276) AND cmdCode = '2605'")

In [0]:
display(de_2605)

In [0]:
# Load data
from pyspark.sql import functions as F
trade = spark.table("bronze_trade_flows")
hs_codes = spark.table("bronze_hs_codes")
countries = spark.table("bronze_country_un_comtrade")

# Rename columns in trade
column_map = {
    'typeCode' : 'trade_type',
    'freqCode' : 'frequency',
    'flowCode' : 'flow_code',
    'period' : 'date',
    'reporterCode' : 'source_code',
    'partnerCode' : 'target_code',
    'cmdCode' : 'hs_code',
    'cifvalue' : 'cif_value',
    'fobvalue' : 'fob_value',
    'primaryValue' : 'primary_value',
    'qty': 'quantity',
    'netWgt' : 'net_weight'
}
trade = trade.select(
    *[
        F.col(old).alias(new)
        for old, new in column_map.items()
    ]
)

In [0]:
# Split out import and export flows and flatten each
from pyspark.sql import functions as F
import_flows = ['M', 'FM', 'MIP', 'MOP', 'RM']
export_flows = ['X', 'DX', 'RX', 'XIP', 'XOP']

imports = trade[trade['flow_code'].isin(import_flows)]
exports = trade[trade['flow_code'].isin(export_flows)]

# Flip source and target labels for imports
imports = (
    imports.withColumnRenamed("source_code", "tmp_code")
      .withColumnRenamed("target_code", "source_code")
      .withColumnRenamed("tmp_code", "target_code")
)

# Pivot on flowCode
metrics = [
    "primary_value",
    "fob_value",
    "cif_value",
    "quantity",
    "net_weight"
]

# Create a struct of all measures
df_struct_imports = imports.withColumn(
    "measures",
    F.struct(*[F.col(c) for c in metrics])
)
df_struct_exports = exports.withColumn(
    "measures",
    F.struct(*[F.col(c) for c in metrics])
)
# Pivot flowCode
imports_pivoted = (
    df_struct_imports
    .groupBy("source_code", "target_code", "hs_code", "date")
    .pivot("flow_code", import_flows)
    .agg(F.first("measures"))
)
exports_pivoted = (
    df_struct_exports
    .groupBy("source_code", "target_code", "hs_code", "date")
    .pivot("flow_code", export_flows)
    .agg(F.first("measures"))
)

# Flatten the structs into columns
flat_imports = imports_pivoted.select(
    "source_code",
    "target_code",
    'hs_code',
    'date',
    *[
        F.col(f"`{flow}`.`{metric}`").alias(f"{flow}_{metric}")
        for flow in import_flows
        for metric in metrics
    ]
)

flat_exports = exports_pivoted.select(
    "source_code",
    "target_code",
    'hs_code',
    'date',
    *[
        F.col(f"`{flow}`.`{metric}`").alias(f"{flow}_{metric}")
        for flow in export_flows
        for metric in metrics
    ]
)




In [0]:
# Combine full flows and clean

full_flows = flat_imports.join(
    flat_exports,
    on=["hs_code", "source_code", "target_code", "date"],
    how="full"
)
full_flows = (
    full_flows
    .withColumn(
        "primary_value",
        F.greatest("M_primary_value", "X_primary_value")
    )
    .withColumn(
        "cif_value",
        F.greatest("M_cif_value", "X_cif_value")
    )
    .withColumn(
        "fob_value",
        F.greatest("M_fob_value", "X_fob_value")
    )
    .withColumn(
        "quantity",
        F.greatest("M_quantity", "X_quantity")
    )
    .withColumn(
        "net_weight",
        F.greatest("M_net_weight", "X_net_weight")
    )
)

flow_names = {
    "M": "import",
    "X": "export",
    "DX": "domestic_export",
    "FM": "foreign_import",
    "MIP": "import_after_inward_processing",
    "MOP": "import_after_outward_processing",
    "RM": "reimport",
    "RX": "reexport",
    "XIP": "export_after_inward_processing",
    "XOP": "export_for_outward_processing",
}


new_cols = []

for col_name in full_flows.columns:
    new_name = col_name

    for code, description in flow_names.items():
        prefix = f"{code}_"

        if col_name.startswith(prefix):
            new_name = col_name.replace(prefix, f"{description}_", 1)
            break

    new_cols.append(F.col(col_name).alias(new_name))

full_flows = full_flows.select(*new_cols)

priority_cols = [
    "date",
    "hs_code",
    "source_code",
    "target_code",
    "primary_value",
    "cif_value",
    "fob_value",
    "quantity",
    "net_weight"
]

other_cols = [c for c in full_flows.columns if c not in priority_cols]

full_flows = full_flows.select(*(priority_cols + other_cols))

In [0]:
# Extract HS code descriptions
from pyspark.sql import functions as F
from pyspark.sql.types import *

schema = StructType([
    StructField("id", StringType()),
    StructField("text", StringType()),
    StructField("parent", StringType()),
    StructField("isLeaf", StringType()),
    StructField("aggrLevel", IntegerType()),
    StructField("standardUnitAbbr", StringType())
])

hs_code = (
    hs_codes
    .withColumn(
        "parsed",
        F.from_json(
            F.regexp_replace("raw_json", "'", '"'),
            schema
        )
    )
    .select(
        F.col("code").alias("hs_code"),
         F.trim(
             F.regexp_replace(
            F.col("parsed.text"),
            r"^.*-\s",
            ""
             )
        ).alias("hs_code_description")
    )
)

display(hs_code)

In [0]:
# Join in country names
source = countries.select(
    F.col("id").alias("source_code"),
    F.col("text").alias("source_country_name"),
    F.col("reporterCodeIsoAlpha2").alias("source_country_code_iso_2"),
    F.col("reporterCodeIsoAlpha3").alias("source_country_code_iso_3")
)

target = countries.select(
    F.col("id").alias("target_code"),
    F.col("text").alias("target_country_name"),
    F.col("reporterCodeIsoAlpha2").alias("target_country_code_iso_2"),
    F.col("reporterCodeIsoAlpha3").alias("target_country_code_iso_3")
)

enriched = (
    full_flows
    .join(source, on="source_code", how="left")
    .join(target, on="target_code", how="left")
    .join(hs_code, on="hs_code", how="left")
)



In [0]:
display(priority_cols)

In [0]:
display(enriched.filter('source_code == 490'))

In [0]:
priority_cols = [ "date", "hs_code", "hs_code_description", "source_code", "source_country_code_iso_2", "source_country_name", "target_code", "target_country_code_iso_2", "target_country_name", "primary_value", "cif_value", "fob_value", "quantity", "net_weight" ] 
other_cols = [c for c in enriched.columns if c not in priority_cols] 
enriched = enriched.select(*(priority_cols + other_cols)) 
display(enriched)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Step 1: drop exact (source, target, year, hs_code) duplicate rows ---
#key_cols = ["source_code", "target_code", "date", "hs_code"]

#dupe_counts = enriched.groupBy(*key_cols).count().filter("count > 1")
#print(f"Found {dupe_counts.count():,} (source, target, year, hs_code) combos with duplicate rows")

# dropDuplicates keeps one row per key arbitrarily when the duplicate rows are
# genuinely identical. If duplicate rows can DISAGREE on values (e.g. two
# different loads of the same key with different primary_value), and you have
# a load timestamp column somewhere, swap this for a window + row_number()
# ordered by that timestamp so you deterministically keep the latest load.
trade_deduped = enriched#.dropDuplicates(key_cols)

#print(f"Rows before: {enriched.count():,}  |  after exact-dup removal: {trade_deduped.count():,}")

# --- Step 2: identify same trade content reported under more than one year,
# and keep only the most recent year ---
# "Same content" = same source/target/hs_code AND matching primary_value + quantity
# across different `date` values. Adjust value_cols if you want a stricter or
# looser definition of "the same record."
content_key = ["source_code", "target_code", "hs_code"]

cross_year_dupes = (
    trade_deduped.groupBy(*content_key)
    .agg(F.countDistinct("date").alias("n_years"))
    .filter("n_years > 1")
)
print(f"Found {cross_year_dupes.count():,} (source, target, hs_code) combos "
      f"repeated across multiple years")

w_year = Window.partitionBy(*content_key).orderBy(F.col("date").desc())

trade_final = (
    trade_deduped
    .withColumn("_rn_year", F.row_number().over(w_year))
    .filter("_rn_year = 1")
    .drop("_rn_year")
)

print(f"Rows before cross-year dedup: {trade_deduped.count():,}  |  "
      f"after: {trade_final.count():,}")

In [0]:
display(trade_final.filter(trade_final.source_code == 490))

In [0]:
%sql
SELECT count(*) FROM silver_trade_flow

In [0]:
%sql
SELECT * FROM silver_trade_flow WHERE target_code = '490' ORDER BY primary_value DESC LIMIT 100

In [0]:
trade_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_trade_flow")